## Descarga de modelos en Ollama

Ollama gestiona los modelos de lenguaje de forma similar a como Docker gestiona contenedores. Antes de poder utilizar un modelo, es necesario **descargarlo** al sistema local. Esta operación solo se realiza una vez por modelo.

Para descargar un modelo, se utiliza el comando `ollama pull` seguido del nombre y la etiqueta del modelo:

In [ ]:
%%bash
ollama pull gemma3:1b

Este comando descarga el modelo **Gemma 3** en su variante de 1 billón de parámetros. Los modelos pequeños como este son ideales para desarrollo y pruebas, ya que consumen menos recursos y responden más rápido.

> Los modelos se almacenan en `~/.ollama/models` en Linux/macOS y en `%LOCALAPPDATA%\ollama\models` en Windows. Puedes verificar el espacio disponible antes de descargar modelos grandes.

Para ver todos los modelos descargados en tu sistema:

In [ ]:
%%bash
ollama list

El siguiente diagrama ilustra el flujo de descarga y almacenamiento de modelos:

```mermaid
flowchart LR
    subgraph Internet
        Registry["Ollama Registry"]
    end
    
    subgraph Local["Sistema local"]
        CLI["ollama pull"]
        Storage["Almacenamiento de modelos"]
    end
    
    Registry -->|Descarga| CLI
    CLI -->|Guarda| Storage
    
    style Registry fill:#3b82f6,color:#fff
    style Storage fill:#10b981,color:#fff
```

### Descargar gpt-oss

El modelo **gpt-oss** es un modelo open-weight de OpenAI diseñado para razonamiento avanzado y tareas complejas. Una de sus características distintivas es el acceso completo al proceso de razonamiento (chain-of-thought).

In [ ]:
%%bash
ollama pull gpt-oss:20b

El modelo gpt-oss:20b ocupa aproximadamente 14 GB y requiere al menos 16 GB de memoria para ejecutarse. Utiliza cuantización en formato MXFP4, lo que permite ejecutar un modelo de 20 billones de parámetros en hardware de consumo.

> Si tu equipo no dispone de suficiente memoria para gpt-oss:20b, puedes seguir los ejemplos con gemma3:1b y observar las diferencias en la documentación.

## Instanciar ChatOllama

La clase **ChatOllama** del paquete `langchain-ollama` es el punto de entrada para interactuar con modelos de Ollama desde Python. Esta clase implementa la interfaz estándar de modelos de chat de LangChain.

In [ ]:
from langchain_ollama import ChatOllama

# Instanciar el modelo
modelo = ChatOllama(model="gemma3:1b")

El parámetro `model` especifica qué modelo de Ollama utilizar. Debe coincidir exactamente con el nombre mostrado por `ollama list`.

```mermaid
flowchart TB
    subgraph Python["Aplicación Python"]
        ChatOllama["ChatOllama"]
    end
    
    subgraph Ollama["Servidor Ollama"]
        API["API REST :11434"]
        Model["Modelo cargado"]
    end
    
    ChatOllama -->|HTTP| API
    API --> Model
    Model -->|Respuesta| API
    API -->|JSON| ChatOllama
    
    style ChatOllama fill:#3b82f6,color:#fff
    style Model fill:#10b981,color:#fff
```

### Verificar la conexión con el modelo

Antes de realizar inferencias complejas, es recomendable verificar que el modelo está correctamente configurado:

In [ ]:
from langchain_ollama import ChatOllama

modelo = ChatOllama(model="gemma3:1b")

# Verificar que el modelo responde
respuesta = modelo.invoke("Hola")
print(respuesta.content)

Si el modelo no está descargado o el servidor Ollama no está activo, este código lanzará una excepción con información sobre el problema.

## Invocar Gemma e interpretar la respuesta

El método `invoke` es la forma más directa de obtener una respuesta del modelo. Acepta una cadena de texto como entrada y devuelve un objeto **AIMessage** con la respuesta generada.

In [ ]:
from langchain_ollama import ChatOllama

modelo = ChatOllama(model="gemma3:1b")

# Invocar con texto simple
respuesta = modelo.invoke("¿Cuál es la capital de Francia?")

# El objeto respuesta es de tipo AIMessage
print(type(respuesta))
# <class 'langchain_core.messages.ai.AIMessage'>

El objeto **AIMessage** encapsula toda la información devuelta por el modelo. El atributo más importante es `content`, que contiene el texto generado:

In [ ]:
from langchain_ollama import ChatOllama

modelo = ChatOllama(model="gemma3:1b")
respuesta = modelo.invoke("Explica qué es Python en una frase")

# Acceder al contenido de la respuesta
print(respuesta.content)

La salida será algo similar a:

```
Python es un lenguaje de programación interpretado, de alto nivel y propósito general, conocido por su sintaxis clara y legible.
```

> El atributo `content` siempre devuelve una cadena de texto con la respuesta del modelo. Este es el dato principal que utilizarás en la mayoría de aplicaciones.

### Metadatos de la respuesta

El objeto AIMessage también incluye información adicional sobre el proceso de generación en el atributo `response_metadata`:

In [ ]:
from langchain_ollama import ChatOllama

modelo = ChatOllama(model="gemma3:1b")
respuesta = modelo.invoke("¿Qué es una API?")

# Metadatos de la generación
print(f"Modelo usado: {respuesta.response_metadata.get('model')}")
print(f"Razón de parada: {respuesta.response_metadata.get('done_reason')}")

# Información de tokens
print(f"Tokens de entrada: {respuesta.usage_metadata.get('input_tokens')}")
print(f"Tokens de salida: {respuesta.usage_metadata.get('output_tokens')}")
print(f"Total tokens: {respuesta.usage_metadata.get('total_tokens')}")

El campo `done_reason` indica por qué el modelo dejó de generar texto. El valor `stop` significa que el modelo completó su respuesta de forma natural.

```mermaid
flowchart TB
    subgraph AIMessage["Objeto AIMessage"]
        Content["content: str"]
        ResponseMeta["response_metadata: dict"]
        UsageMeta["usage_metadata: dict"]
    end
    
    ResponseMeta --> Model["model"]
    ResponseMeta --> Done["done_reason"]
    UsageMeta --> Input["input_tokens"]
    UsageMeta --> Output["output_tokens"]
    
    style AIMessage fill:#3b82f6,color:#fff
    style Content fill:#10b981,color:#fff
```

## Invocar gpt-oss e interpretar el razonamiento

El modelo **gpt-oss** de OpenAI tiene una característica distintiva: expone su proceso de razonamiento completo (chain-of-thought). Esto significa que puedes ver cómo el modelo "piensa" antes de dar su respuesta final.

In [ ]:
from langchain_ollama import ChatOllama

modelo = ChatOllama(model="gpt-oss:20b")

respuesta = modelo.invoke("¿Cuánto es 15 multiplicado por 7?")
print(respuesta.content)

A diferencia de gemma3, la respuesta de gpt-oss incluye el proceso de razonamiento:

```
Voy a calcular 15 multiplicado por 7.

15 × 7 = 105

La respuesta es 105.
```

> El acceso al chain-of-thought de gpt-oss permite depurar y entender mejor cómo el modelo llega a sus conclusiones, lo cual es especialmente útil en tareas de razonamiento complejo.

### Comparar respuestas entre modelos

Una forma práctica de entender las diferencias entre modelos es hacer la misma pregunta a ambos:

In [ ]:
from langchain_ollama import ChatOllama

# Instanciar ambos modelos
gemma = ChatOllama(model="gemma3:1b")
gpt_oss = ChatOllama(model="gpt-oss:20b")

pregunta = "Si tengo 3 manzanas y me dan el doble, ¿cuántas tengo?"

# Respuesta de Gemma
print("=== Gemma 3 ===")
respuesta_gemma = gemma.invoke(pregunta)
print(respuesta_gemma.content)

# Respuesta de gpt-oss con razonamiento
print("\n=== gpt-oss ===")
respuesta_gpt = gpt_oss.invoke(pregunta)
print(respuesta_gpt.content)

La respuesta de **Gemma** será directa:

```
Tendrías 6 manzanas.
```

La respuesta de **gpt-oss** mostrará el razonamiento:

```
Analicemos el problema paso a paso:
1. Empiezo con 3 manzanas
2. Me dan el doble de lo que tengo, es decir: 3 × 2 = 6 manzanas adicionales
3. Total: 3 + 6 = 9 manzanas

La respuesta es 9 manzanas.
```

Este ejemplo ilustra cómo diferentes modelos pueden interpretar la misma pregunta de formas distintas. Gpt-oss tiende a desglosar el problema y mostrar cada paso, mientras que modelos más pequeños dan respuestas más concisas.

> El modelo gpt-oss es especialmente útil cuando necesitas verificar el razonamiento detrás de una respuesta o cuando trabajas con problemas que requieren múltiples pasos de lógica.

### Estructura del AIMessage con gpt-oss

El objeto AIMessage devuelto por gpt-oss tiene la misma estructura que el de cualquier otro modelo, pero el contenido incluye tanto el razonamiento como la respuesta:

In [ ]:
from langchain_ollama import ChatOllama

modelo = ChatOllama(model="gpt-oss:20b")
respuesta = modelo.invoke("¿Por qué el cielo es azul?")

# El content incluye todo el proceso de pensamiento
print(f"Longitud de la respuesta: {len(respuesta.content)} caracteres")
print(f"Tokens generados: {respuesta.usage_metadata.get('output_tokens')}")

Las respuestas de gpt-oss suelen ser más largas debido al razonamiento incluido, lo que se refleja en un mayor número de tokens de salida. Esto es importante tenerlo en cuenta al planificar el uso de recursos.
